# Day 4 — Cost of the schedule: spread, impact, implementation shortfall

Scope: the Day 3 timing engine is carried over **unchanged** — month-end formation, execution starting only after a lag of L business days, names worked in liquidity order under a daily participation cap — and every fill it produces is now priced. Output is **cost only**: implementation shortfall against the month-end decision price, split into delay, spread, impact and opportunity. No intraday split and no clustering indicator; those still attach later. AUM 1000億, same lag grid 0-20bd, same cap grid 2-30%.

Day 3 left one question open in as many words — liquid-first buys the liquid quintile time and costs the illiquid tail delay, and *"whether that trade is worth making is a cost question, not a timing one, so it is left to the next pass"*. This is that pass.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

DATA_DIR = Path("/home/ubuntu/work/data")
OUT_DIR  = Path("/home/ubuntu/work/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)
UNIVERSE_FILE  = DATA_DIR / "jp_top500_universe_201001_202607.parquet"
RETURNS_FILE   = DATA_DIR / "daily_returns_20100104_20260729.parquet"
PORTFOLIO_FILE = DATA_DIR / "jp_top500_monthly_bpr_portfolios_201001_202606.csv"

COLS = dict(ret_date="base_ymd", ret_code="stock_code", ret_value="daily_return",
            ret_price="price_close", ret_volume="volume",
            uni_date="date_eom", uni_code="code", uni_cap="mkt_cap",
            prt_date="DATE_EOM", prt_code="CODE", prt_weight="WEIGHT")
OI = ["#0072B2", "#009E73", "#E69F00", "#D55E00", "#56B4E9", "#CC79A7", "#F0E442", "#000000"]

# ---- Day 3 settings, unchanged. The timing engine in section 3 is copied verbatim. ----
AUM            = 100_000_000_000          # 1000億
LAG_GRID       = [0, 2, 5, 7, 10, 13, 15, 17, 20]        # business days after the month-end decision
CAP_GRID       = [0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]   # own share of total daily volume
BASE_LAG, BASE_CAP = 2, 0.10
ADV_WIN        = 20                       # trailing window for ADV and for the liquidity quintiles
MAX_EXEC_DAYS  = 500                      # give up on a name after this many bd from its own start
WAVE_WAIT_BD   = 20                       # a liquidity stage waits at most this long for the previous stage
BUCKET_NAMES   = ["Q1 illiquid", "Q2", "Q3", "Q4", "Q5 liquid"]   # index 0..4, 4 = most liquid
RNG = np.random.default_rng(7)

# ---- Day 4 additions: the cost model --------------------------------------------------
VOL_WIN        = 20        # trailing window for the daily return vol that scales impact
ETA            = 0.5       # square-root impact coefficient: cost = ETA * sigma * sqrt(rate)
ETA_GRID       = [0.25, 0.5, 1.0]        # reported as a sensitivity, not as three results
SPREAD_TICKS   = 1.0       # assumed quoted spread in ticks; a schedule pays half of it
FINE_TICK_FROM = pd.Timestamp("2023-06-05")   # TSE fine tick grid extended to all of TOPIX500
BREAK_TOL      = 0.01      # |daily_return - close-to-close| above this is an adjustment break
MAX_PANEL_CELLS = 40_000_000   # refuse to allocate panels bigger than this (see section 2)

# TSE yobine (tick size) ladders as (upper bound inclusive, tick), both in deci-yen.
# Source: this project's own frozen tables (s0_common.YOBINE_*), cross-checked there
# against ticks inferred from the tape.
YOBINE_GENERAL = [(30_000, 10), (50_000, 50), (300_000, 100), (500_000, 500),
                  (3_000_000, 1_000), (5_000_000, 5_000), (30_000_000, 10_000),
                  (50_000_000, 50_000), (300_000_000, 100_000), (500_000_000, 500_000),
                  (None, 1_000_000)]
YOBINE_TOPIX500 = [(10_000, 1), (30_000, 5), (100_000, 10), (300_000, 50),
                   (1_000_000, 100), (3_000_000, 500), (10_000_000, 1_000),
                   (30_000_000, 5_000), (100_000_000, 10_000), (300_000_000, 50_000),
                   (None, 100_000)]

SYNTH = not (UNIVERSE_FILE.exists() and RETURNS_FILE.exists() and PORTFOLIO_FILE.exists())
if SYNTH:
    print(">>> data files not found: generating SYNTHETIC data (plumbing test only, numbers meaningless)")
    sdir = Path("/tmp/day4_synth"); sdir.mkdir(parents=True, exist_ok=True)
    days = pd.bdate_range("2019-01-02", periods=900); codes = np.array([str(1301 + i) for i in range(120)])
    nd, nn = len(days), len(codes)
    rets = RNG.normal(0.0003, 0.02, (nd, nn))
    prices = 1000 * np.exp(RNG.normal(0, 0.5, nn))[None, :] * np.cumprod(1 + rets, axis=0)
    advy = np.exp(RNG.uniform(np.log(8e7), np.log(3e10), nn))
    vol = advy[None, :] * np.exp(RNG.normal(0, 0.4, (nd, nn))) / prices
    pd.DataFrame(dict(base_ymd=np.repeat(days.strftime("%Y%m%d").astype(np.int64), nn), stock_code=np.tile(codes, nd),
                      daily_return=rets.ravel(), price_close=prices.ravel(), volume=vol.ravel())).to_parquet(sdir / "r.parquet")
    eoms = pd.DatetimeIndex(pd.Series(days, index=days).groupby(days.to_period("M")).max())
    dp = {d: i for i, d in enumerate(days)}; sh = np.exp(RNG.normal(17, 1.2, nn)); capm = prices * sh[None, :]
    pd.concat([pd.DataFrame(dict(date_eom=int(e.strftime("%Y%m%d")), code=codes, mkt_cap=capm[dp[e]])) for e in eoms]).to_parquet(sdir / "u.parquet")
    book = prices[0] * np.exp(RNG.normal(0, 0.3, nn)); prt = []
    for e in eoms[:-1]:
        pbr = prices[dp[e]] / book; pick = np.argsort(pbr)[:60]; w = (1 / pbr[pick]); w /= w.sum()
        prt.append(pd.DataFrame(dict(DATE_EOM=e.strftime("%Y-%m-%d"), CODE=codes[pick], WEIGHT=w)))
    pd.concat(prt).to_csv(sdir / "p.csv", index=False)
    UNIVERSE_FILE, RETURNS_FILE, PORTFOLIO_FILE = sdir / "u.parquet", sdir / "r.parquet", sdir / "p.csv"
print(f"SYNTH={SYNTH}  AUM={AUM:,}  lags={LAG_GRID}  caps={[f'{c:.0%}' for c in CAP_GRID]}")
print(f"cost model: ETA={ETA} sqrt-impact, half of a {SPREAD_TICKS:g}-tick spread, vol window {VOL_WIN}d")

## 1. Load and schema audit

Expected schema, taken from the Day 3 run (assert, do not assume):

| file | columns | note |
|---|---|---|
| `daily_returns_*.parquet` | `base_ymd` (int YYYYMMDD), `stock_code`, `daily_return`, `price_close`, `volume` (shares) | ~4,054 trading days x ~5,019 names, 2010-01-04 to 2026-07-29. Yen volume = `price_close * volume`; median across all names ~54.5m yen. |
| `jp_top500_universe_*.parquet` | `date_eom`, `code`, `mkt_cap` | monthly membership, used here only to define the liquidity quintiles cross-sectionally |
| `jp_top500_monthly_bpr_portfolios_*.csv` | `DATE_EOM`, `CODE`, `WEIGHT` | ~198 month-ends x ~100 names, weights renormalised to 1 per date |

Codes are normalised to string, stripped of a trailing `.0`, uppercased. Date columns that arrive as integers are parsed from their string form.

Day 3 read only `price_close * volume` and could stop there. Pricing fills needs two more columns out of the same file, and neither is safe to take on trust:

- **`daily_return`** drives every basis-point number below, so its units are **detected, not assumed** (section 2): a silent percent-versus-decimal mix-up would scale the whole notebook by 100.
- **`price_close`** is used only for its *level*, to place a name in the right tick band. Every shortfall ratio is taken off the compounded return series instead, because an unadjusted close is not split-safe — a 1:2 split would otherwise book a -5,000bp "delay cost" on the day it happened. Check B1 counts how far the two disagree.

The load below is deliberately one pass over the returns file: one frame is built, one dedupe is done, and the raw frame is dropped before anything is reshaped.

In [ ]:
def read(p: Path) -> pd.DataFrame:
    return pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)

def _dates(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s.astype(str)) if str(s.dtype).startswith(("int", "Int")) else pd.to_datetime(s)

def _codes(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.upper()

def tidy(p: Path, keys, name, dedupe=False) -> pd.DataFrame:
    dc, cc, vc = keys
    df = read(p)
    missing = [c for c in keys if c not in df.columns]
    assert not missing, f"{p.name}: missing {missing}. Present: {list(df.columns)}"
    out = pd.DataFrame({"date": _dates(df[dc]), "code": _codes(df[cc]), name: pd.to_numeric(df[vc], errors="coerce")}).dropna()
    return out.groupby(["date", "code"], as_index=False)[name].sum() if dedupe else out

_r = read(RETURNS_FILE)
print("returns file dtypes:"); print(_r.dtypes.to_string())
_need = [COLS[k] for k in ("ret_date", "ret_code", "ret_value", "ret_price", "ret_volume")]
_miss = [c for c in _need if c not in _r.columns]
assert not _miss, f"{RETURNS_FILE.name}: missing {_miss}. Present: {list(_r.columns)}"
_px = pd.to_numeric(_r[COLS["ret_price"]], errors="coerce")
_vo = pd.to_numeric(_r[COLS["ret_volume"]], errors="coerce")
TAPE = pd.DataFrame({"date": _dates(_r[COLS["ret_date"]]), "code": _codes(_r[COLS["ret_code"]]),
                     "yen_volume": _px * _vo, "px": _px,
                     "ret": pd.to_numeric(_r[COLS["ret_value"]], errors="coerce")}).dropna(subset=["date", "code"])
del _r, _px, _vo
TAPE = TAPE.groupby(["date", "code"], as_index=False).agg(yen_volume=("yen_volume", "sum"),
                                                          px=("px", "last"), ret=("ret", "last"))
TAPE["yen_volume"] = TAPE["yen_volume"].fillna(0.0)

UNI = tidy(UNIVERSE_FILE, (COLS["uni_date"], COLS["uni_code"], COLS["uni_cap"]), "cap")
PRT = tidy(PORTFOLIO_FILE, (COLS["prt_date"], COLS["prt_code"], COLS["prt_weight"]), "weight", dedupe=True)

CAL = pd.DatetimeIndex(np.sort(TAPE["date"].unique()))
assert len(CAL) > 100 and CAL.is_monotonic_increasing
print(f"\ncalendar : {len(CAL)} trading days, {CAL.min().date()} -> {CAL.max().date()}")
print(f"tape     : {TAPE['code'].nunique()} names, {len(TAPE):,} name-days, "
      f"median daily yen volume {TAPE['yen_volume'].median():,.0f}")
print(f"universe : {UNI['date'].nunique()} month-ends x {UNI.groupby('date').size().median():.0f} names")
print(f"portfolio: {PRT['date'].nunique()} month-ends x {PRT.groupby('date').size().median():.0f} names, "
      f"weight sum per date min/max {PRT.groupby('date')['weight'].sum().min():.4f}/{PRT.groupby('date')['weight'].sum().max():.4f}")
assert (PRT["weight"] >= 0).all(), "negative portfolio weights: long-only assumption broken"
assert (TAPE["yen_volume"] >= 0).all()

## 2. Panels: volume, price, return, ADV, volatility, liquidity quintiles

Day 3's panels are rebuilt exactly, and three are added. Everything the cost engine needs is read off cumulative sums in closed form, so there is still **no day loop anywhere**: the discipline that makes the timing engine cheap is what makes the pricing engine cheap.

- `VOLMAT[t, i]` yen volume, `CUMV0` its cumulative sum with a leading zero row (Day 3).
- `ADVMAT[t, i]` = trailing `ADV_WIN`-day median yen volume, **shifted one day**, so a decision date only ever sees data strictly before it (Day 3).
- **`PREL[t, i]`** = compounded total-return index, `cumprod(1 + daily_return)`. Every shortfall ratio is taken off this, never off raw closes.
- **`CUMVP0`** = cumulative sum of `VOLMAT * PREL`, again with a leading zero row. This is the new piece: it turns "value-weighted execution price across a name's whole trading window" into two array lookups instead of a loop over days. Verified against a brute-force day-by-day fill in B2.
- **`SIG_DEC`, `TICK_DEC`, `SPRD_DEC`** are `(n_formations, n_names)` slices taken at **decision dates only** — trailing return volatility (shifted one day), the yobine tick for the name's price band, and the resulting half-spread in bps. Keeping the decision rows and dropping the full panels is what holds memory flat.

The tape is cut to `CODES` before any reshape, and panel size is asserted against `MAX_PANEL_CELLS` **before** anything is allocated. `CODES` is the universe joined to the tape, a few hundred names; if a schema change ever widened it to every name on the file, the pivot would quietly try to allocate tens of gigabytes and take the kernel with it. That is a loud failure now.

`RET_SCALE` is set here rather than assumed: the median absolute `daily_return` is compared with the median absolute close-to-close move, and a ratio near 100 means the column arrived in percent.

Liquidity quintiles are cut cross-sectionally at each month-end on the universe members' trailing ADV, then every portfolio name is assigned against those cut points. Bucket 4 = most liquid; a name with no ADV history gets bucket 0 (most conservative).

In [ ]:
CODES = np.array(sorted(set(PRT["code"]) | set(UNI["code"])))
CODES = np.array([c for c in CODES if c in set(TAPE["code"])])
CPOS  = pd.Series(np.arange(len(CODES)), index=CODES); NC = len(CODES)
DPOS  = pd.Series(np.arange(len(CAL)), index=CAL)

_cells = len(CAL) * NC
assert _cells <= MAX_PANEL_CELLS, (
    f"panel would be {len(CAL)} days x {NC} names = {_cells:,} cells "
    f"({_cells * 8 / 1e9:.1f} GB per matrix), over the {MAX_PANEL_CELLS:,} cell limit. "
    "CODES has been widened beyond the universe: check the join keys before re-running.")
print(f"panels: {len(CAL)} days x {NC} names = {_cells:,} cells, {_cells * 8 / 1e6:.0f} MB per matrix")

_n0 = len(TAPE)
TAPE = TAPE[TAPE["code"].isin(set(CODES))]      # nothing outside the universe is used again
print(f"tape cut to the universe before any reshape: {_n0:,} -> {len(TAPE):,} name-days")
VOLW = TAPE.pivot_table(index="date", columns="code", values="yen_volume", aggfunc="first").reindex(CAL).reindex(columns=CODES)
PXW  = TAPE.pivot_table(index="date", columns="code", values="px", aggfunc="first").reindex(CAL).reindex(columns=CODES)
RTW  = TAPE.pivot_table(index="date", columns="code", values="ret", aggfunc="first").reindex(CAL).reindex(columns=CODES)

VOLMAT = np.nan_to_num(VOLW.to_numpy(dtype=float), nan=0.0)
CUMV0  = np.asfortranarray(np.vstack([np.zeros((1, NC)), np.cumsum(VOLMAT, axis=0)]))   # (T+1, NC)
ADVMAT = VOLW.where(VOLW > 0).rolling(ADV_WIN, min_periods=5).median().shift(1).to_numpy(dtype=float)
PXMAT  = PXW.to_numpy(dtype=float)
RAWRET = RTW.to_numpy(dtype=float)

# return units: detect, do not assume
_c2c = PXMAT[1:] / np.where(PXMAT[:-1] > 0, PXMAT[:-1], np.nan) - 1.0
_ratio = float(np.nanmedian(np.abs(RAWRET[1:])) / max(np.nanmedian(np.abs(_c2c)), 1e-12))
RET_SCALE = 0.01 if _ratio > 20 else 1.0
print(f"return units: median|daily_return| / median|close-to-close| = {_ratio:.3f} -> "
      f"{'PERCENT, scaled by 1/100' if RET_SCALE != 1.0 else 'DECIMAL, used as is'}")

RETMAT = RAWRET * RET_SCALE
PREL   = np.cumprod(1.0 + np.nan_to_num(RETMAT, nan=0.0), axis=0)
CUMVP0 = np.asfortranarray(np.vstack([np.zeros((1, NC)), np.cumsum(VOLMAT * PREL, axis=0)]))
SIGMAT = (RTW * RET_SCALE).rolling(VOL_WIN, min_periods=5).std().shift(1).to_numpy(dtype=float)

# B1 evidence, gathered while both series are still in memory
_both = np.isfinite(RETMAT[1:]) & np.isfinite(_c2c)
_gap  = np.abs(RETMAT[1:] - _c2c)
RET_VS_PX = dict(n=int(_both.sum()),
                 n_break=int((_both & (_gap > BREAK_TOL)).sum()),
                 worst=float(np.nanmax(np.where(_both, _gap, np.nan))) if _both.any() else np.nan,
                 median_gap=float(np.nanmedian(np.where(_both, _gap, np.nan))) if _both.any() else np.nan)
del _c2c, _both, _gap, RAWRET

def snap(dates) -> pd.DatetimeIndex:
    pos = CAL.get_indexer(pd.DatetimeIndex(pd.to_datetime(dates)), method="ffill")
    assert (pos >= 0).all(), "a date falls before the start of the trading calendar"
    return CAL[pos]

def weight_vectors(w: pd.DataFrame) -> dict:
    out = {}
    for d, g in w.groupby("date"):
        v = np.zeros(NC)
        j = CPOS.reindex(g["code"]).to_numpy(dtype=float); ok = ~np.isnan(j)
        v[j[ok].astype(int)] = g["weight"].to_numpy()[ok]
        if v.sum() > 0:
            out[snap([d])[0]] = v / v.sum()
    return out

WVEC  = weight_vectors(PRT)
FORMS = pd.DatetimeIndex(sorted(WVEC.keys()))
FIDX  = np.array([int(DPOS[f]) for f in FORMS])

UNI_D = pd.DatetimeIndex(np.sort(UNI["date"].unique()))
UNI_MEM = {d: CPOS.reindex(g["code"]).dropna().to_numpy(dtype=int) for d, g in UNI.groupby("date")}
def universe_at(f):
    k = UNI_D.searchsorted(f, side="right") - 1
    return UNI_MEM[UNI_D[k]] if k >= 0 else np.arange(NC)

BUCKET = {}
for f in FORMS:
    adv = ADVMAT[int(DPOS[f])]
    mem = universe_at(f); mem = mem[np.isfinite(adv[mem]) & (adv[mem] > 0)]
    b = np.zeros(NC, dtype=int)
    if mem.size >= 25:
        cuts = np.quantile(adv[mem], [0.2, 0.4, 0.6, 0.8])
        ok = np.isfinite(adv) & (adv > 0)
        b[ok] = np.searchsorted(cuts, adv[ok], side="right")
    BUCKET[f] = np.clip(b, 0, 4)

def tick_yen(px_yen, fine: bool):
    """Tick size in yen for an array of prices, from the TSE yobine ladder."""
    table = YOBINE_TOPIX500 if fine else YOBINE_GENERAL
    uppers = np.array([u for u, _ in table[:-1]], dtype=float)      # deci-yen, ascending
    ticks  = np.array([t for _, t in table], dtype=float)           # deci-yen
    p = np.asarray(px_yen, dtype=float)
    idx = np.searchsorted(uppers, np.nan_to_num(p, nan=0.0) * 10.0, side="left")
    return np.where(np.isfinite(p) & (p > 0), ticks[idx] / 10.0, np.nan)

# decision-date slices, then drop the full panels that are no longer needed
ADV_DEC  = ADVMAT[FIDX]
SIG_DEC  = SIGMAT[FIDX]
PX_DEC   = PXMAT[FIDX]
_fine    = np.asarray(FORMS >= FINE_TICK_FROM)
TICK_DEC = np.vstack([tick_yen(PX_DEC[k], bool(_fine[k])) for k in range(len(FORMS))])
SPRD_DEC = 0.5 * SPREAD_TICKS * TICK_DEC / np.where(PX_DEC > 0, PX_DEC, np.nan) * 1e4   # bps, half-spread
del VOLW, PXW, RTW, PXMAT, RETMAT, SIGMAT, ADVMAT, TAPE

kf = len(FORMS) // 2; f = FORMS[kf]
adv, bk = ADV_DEC[kf], BUCKET[f]
print(f"\nliquidity quintiles at {f.date()} (trailing {ADV_WIN}d median yen volume):")
for q in range(5):
    sel = (bk == q) & np.isfinite(adv)
    print(f"  {BUCKET_NAMES[q]:<12} n={sel.sum():4d}  median ADV {np.nanmedian(adv[sel]):>18,.0f}"
          f"  half-spread {np.nanmedian(SPRD_DEC[kf][sel]):5.2f}bp"
          f"  sigma {np.nanmedian(SIG_DEC[kf][sel]) * 100:5.2f}%/d")
print(f"\n{len(FORMS)} formation dates; the fine tick grid applies from {FINE_TICK_FROM.date()} "
      f"({int(_fine.sum())} of {len(FORMS)} month-ends)")

## 3. Cost model and engine

The timing engine (`shift_bd_idx`, `completion_idx`, `month_paths`) is Day 3's, copied without edits: on any day the fund takes at most `cap` of that day's total volume including its own print, so a target trade `X` needs cumulative market volume `X*(1-cap)/cap` from its start day, and the completion day is a binary search on `CUMV0`.

What is new is that the same two cumulative columns also give the **price** that schedule achieves. A name starting at `s` and completing at `e` takes `f = cap/(1-cap)` of each day's printed volume on days `s..e-1` and whatever is left on day `e`, so

```
filled_before_last = f * (CUMV0[e] - CUMV0[s])
last_slice         = X - filled_before_last
VWAP               = ( f * (CUMVP0[e] - CUMVP0[s]) + last_slice * PREL[e] ) / X
```

exactly, and with no iteration over days. A name that never completes is handled by the same three lines with `e` set to its cutoff day, which leaves `X - filled` unbought.

Four components, all signed so **positive is a cost**, all quoted in basis points of the name's *target* notional so that they add:

| component | formula | what it is |
|---|---|---|
| delay | `side * (VWAP / PREL[decision] - 1)` | the price moved while the fund waited out the lag, queued behind more liquid quintiles, and worked the order |
| spread | `0.5 * SPREAD_TICKS * tick / price` | half of an assumed one-tick quote, tick from the TSE yobine ladder |
| impact | `ETA * sigma * sqrt(rate)` | square-root law on the participation rate; `sigma` is trailing daily vol at the decision date. `rate` is `cap` on every full day by construction, and `last_slice/(last_slice + volume)` on the final partial day |
| opportunity | `side * (PREL[cutoff] / PREL[decision] - 1) * unfilled / X` | what the part that never got bought did to the paper portfolio |

`side` is `+1` for a buy and `-1` for a sell. Delay, spread and impact are earned on the filled notional and scaled by `filled/X`, so the four terms sum to the shortfall on the target — which is what B5 asserts.

Two things worth being explicit about before any of it is read. **Spread and impact are model, not measurement**: there is no quote data in this file, so they are exactly as good as `SPREAD_TICKS` and `ETA`, and section 6 reports the `ETA` sensitivity rather than burying it. **Delay is measurement, not model**: it is realised price drift, so its mean is a small number sitting on a very large standard deviation, and it is carried with a month-clustered standard error everywhere below. A delay number without its error bar is not a result.

In [ ]:
# ---------------- Day 3 timing engine, verbatim ----------------
def shift_bd_idx(f, n: int):
    i = int(DPOS[snap([f])[0]]) + n
    return i if 0 <= i < len(CAL) else None

def completion_idx(need_vol, cols, start_idx, max_exec=MAX_EXEC_DAYS):
    """Smallest calendar index t >= start_idx at which cumulative market volume since start_idx
    covers need_vol.
      t >= 0 : completed on that day
      -1     : the cap is genuinely binding, not done within max_exec days of a full window
      -2     : the allowed window runs past the end of the sample, so we cannot tell (truncated)"""
    out = np.full(len(cols), -1, dtype=int)
    truncated = start_idx + max_exec > len(CAL) - 1
    lim = min(start_idx + max_exec, len(CAL))
    for k in range(len(cols)):
        col = CUMV0[:, cols[k]]
        idx = int(np.searchsorted(col, col[start_idx] + need_vol[k], side="left"))
        t = idx - 1
        if idx <= len(CAL) - 1 and start_idx <= t < lim:
            out[k] = t
        elif truncated:
            out[k] = -2
    return out

def month_paths(delta_w, bucket_of, i_dec, lag, cap, order="liquid_first",
                max_exec=MAX_EXEC_DAYS, wave_wait=WAVE_WAIT_BD):
    act = np.flatnonzero(delta_w != 0)
    s0 = i_dec + lag
    if act.size == 0 or s0 >= len(CAL):
        return None
    need = np.abs(delta_w[act]) * AUM * (1.0 - cap) / cap
    bk = bucket_of[act]
    start = np.full(act.size, s0, dtype=int); finish = np.full(act.size, -1, dtype=int)
    if order == "parallel":
        groups = [np.arange(act.size)]
    else:
        seq = [4, 3, 2, 1, 0] if order == "liquid_first" else [0, 1, 2, 3, 4]
        groups = [np.flatnonzero(bk == b) for b in seq]
    s = s0
    for g in groups:
        if g.size == 0:
            continue
        start[g] = s
        fin = completion_idx(need[g], act[g], s, max_exec)
        finish[g] = fin
        done = fin >= 0
        stage_end = int(fin[done].max()) if done.any() else min(s + max_exec, len(CAL) - 1)
        s = int(min(stage_end + 1, s + wave_wait, len(CAL) - 1))
    return act, start, finish, bk, need

# ---------------- Day 4 pricing, closed form ----------------
def price_paths(res, delta_w, k, i_dec, cap, eta=ETA, max_exec=MAX_EXEC_DAYS):
    """Price one month's schedule. One row per traded name, costs in bp of target notional."""
    act, start, finish, bk, need = res
    f_ = cap / (1.0 - cap)
    X  = need * f_                                    # target yen per name
    done = finish >= 0
    i_end = np.where(done, np.maximum(finish, start), np.minimum(start + max_exec, len(CAL) - 1))

    vol_pre  = CUMV0[i_end, act] - CUMV0[start, act]        # market volume over [start, i_end-1]
    vp_pre   = CUMVP0[i_end, act] - CUMVP0[start, act]
    exec_pre = f_ * vol_pre
    last_mkt = VOLMAT[i_end, act]
    last_ex  = np.where(done, np.clip(X - exec_pre, 0.0, None), f_ * last_mkt)
    filled   = np.minimum(exec_pre + last_ex, X)
    pos      = filled > 0
    denom    = np.where(pos, filled, 1.0)

    vwap  = np.where(pos, (f_ * vp_pre + last_ex * PREL[i_end, act]) / denom, PREL[start, act])
    p0    = PREL[i_dec, act]
    side  = np.sign(delta_w[act])
    fillr = np.where(X > 0, filled / np.where(X > 0, X, 1.0), 0.0)

    rate_last = np.where(last_mkt + last_ex > 0, last_ex / (last_mkt + last_ex), 0.0)
    sig  = SIG_DEC[k, act]
    imp  = np.where(pos, (exec_pre * eta * sig * np.sqrt(cap) +
                          last_ex * eta * sig * np.sqrt(rate_last)) / denom, 0.0) * 1e4
    half = SPRD_DEC[k, act]

    delay_bps  = side * (vwap / p0 - 1.0) * 1e4 * fillr
    spread_bps = half * fillr
    impact_bps = imp * fillr
    opp_bps    = side * (PREL[i_end, act] / p0 - 1.0) * 1e4 * (1.0 - fillr)

    return pd.DataFrame({
        "decision": CAL[i_dec], "code": CODES[act], "bucket": bk, "side": side,
        "start_offset_bd": start - i_dec,
        "days_exec": np.where(done, i_end - start + 1, np.nan),
        "days_from_decision": np.where(done, i_end - i_dec + 1, np.nan),
        "trade_yen": X, "fill_ratio": fillr, "vwap_rel": vwap, "p0_rel": p0,
        "half_spread_bps": half,
        "delay_bps": delay_bps, "spread_bps": spread_bps, "impact_bps": impact_bps,
        "opportunity_bps": opp_bps,
        "total_is_bps": delay_bps + spread_bps + impact_bps + opp_bps,
        "truncated": finish == -2,
    })

COST_COLS = ["delay_bps", "spread_bps", "impact_bps", "opportunity_bps", "total_is_bps"]

def vw_by(D, col, by, w="trade_yen"):
    """Value-weighted mean of `col` within `by`. Plain sums, so it behaves the same on any pandas."""
    by = [by] if isinstance(by, str) else list(by)
    v = D[col].to_numpy(dtype=float); ww = D[w].to_numpy(dtype=float)
    m = np.isfinite(v) & np.isfinite(ww) & (ww > 0)
    t = pd.DataFrame({k: D[k].to_numpy()[m] for k in by})
    t["num"] = v[m] * ww[m]; t["den"] = ww[m]
    s = t.groupby(by, observed=True).sum()
    return s["num"] / s["den"]

def vw(D, col, w="trade_yen"):
    v = D[col].to_numpy(dtype=float); ww = D[w].to_numpy(dtype=float)
    m = np.isfinite(v) & np.isfinite(ww) & (ww > 0)
    return float(np.average(v[m], weights=ww[m])) if m.any() else np.nan

def run_combo(lag, cap, order="liquid_first", detail=False, eta=ETA):
    """Runs every month. Per-bucket cost and timing, value-weighted by trade size."""
    parts = []
    for m in range(1, len(FORMS)):
        d1 = FORMS[m]; i_dec = int(DPOS[d1])
        dw = WVEC[d1] - WVEC[FORMS[m - 1]]
        res = month_paths(dw, BUCKET[d1], i_dec, lag, cap, order)
        if res is None:
            continue
        parts.append(price_paths(res, dw, m, i_dec, cap, eta))
    D_all = pd.concat(parts, ignore_index=True)
    trunc = D_all.groupby("bucket")["truncated"].mean().reindex(range(5))
    D = D_all[~D_all["truncated"]].copy()          # sample-end cases carry no information
    g = D.groupby("bucket")

    # month-clustered standard error: value-weight within month, then across months
    mo = vw_by(D, "total_is_bps", ["bucket", "decision"]).unstack("bucket")
    se = (mo.std(ddof=1) / np.sqrt(mo.notna().sum())).reindex(range(5))

    S = pd.DataFrame({
        "n": g.size(),
        "mean_start_offset_bd": g["start_offset_bd"].mean(),
        "mean_days_exec": g["days_exec"].mean(),
        "fill_ratio": vw_by(D, "fill_ratio", "bucket").reindex(range(5)),
        "share_unresolved": g["days_exec"].apply(lambda s: s.isna().mean()),
        "share_truncated_by_sample_end": trunc,
        "half_spread_bps": vw_by(D, "half_spread_bps", "bucket").reindex(range(5)),
        **{c: vw_by(D, c, "bucket").reindex(range(5)) for c in COST_COLS},
        "total_is_se": se,
        "median_trade_yen": g["trade_yen"].median(),
        "sum_trade_yen": g["trade_yen"].sum(),
    }).reindex(range(5))
    S["total_is_t"] = S["total_is_bps"] / S["total_is_se"]
    S.index = pd.Index([BUCKET_NAMES[b] for b in S.index], name="bucket")
    S.insert(0, "cap", cap); S.insert(0, "lag_bd", lag); S.insert(0, "order", order)
    return (S, D_all) if detail else S

_t0 = time.time(); _ = run_combo(BASE_LAG, BASE_CAP); _dt = time.time() - _t0
print(f"engine ready: one combination over {len(FORMS) - 1} months takes {_dt:.2f}s, so the "
      f"{len(LAG_GRID) * len(CAP_GRID)}-combination grid in section 6 should take about "
      f"{_dt * len(LAG_GRID) * len(CAP_GRID) / 60:.1f} min")

## 4. Cost audit

Day 3's lag checks (A1-A6) still hold unchanged, because the timing engine is unchanged; A3 is re-run below as B6 because it is the one that would invalidate every cost number if it broke. The new checks are about the pricing:

- **B1** `daily_return` and `price_close` tell the same story. Name-days where they disagree by more than `BREAK_TOL` are counted — those are splits and adjustment breaks, and they are the reason shortfall is measured off returns rather than off closes.
- **B2** the closed-form VWAP equals an explicit day-by-day fill loop on a sample of name-months. This is the check that matters: it is what licenses running the whole grid with no day loop.
- **B3** structural — a name that starts on the decision day and finishes the same day must price at exactly that day's price, so its delay term is zero to the last bit.
- **B4** monotonicity and invariance — impact rises with the cap; the per-unit half-spread is identical for a given name-month across every lag and cap, since it depends only on the tick band at the decision date; the dispersion of the delay term rises with the lag.
- **B5** the decomposition adds up: delay + spread + impact + opportunity = total, to floating tolerance.
- **B6** no lookahead: nothing starts before decision + lag, under every ordering and every lag.
- **B7** one month printed line by line, with dates, prices and the cost split, so it can be eyeballed.

In [ ]:
# B1 returns versus closes
print(f"B1 return/close agreement over {RET_VS_PX['n']:,} name-days: "
      f"median gap {RET_VS_PX['median_gap']:.2e}, worst {RET_VS_PX['worst']:.3f}, "
      f"{RET_VS_PX['n_break']:,} breaks over {BREAK_TOL:.0%} "
      f"({RET_VS_PX['n_break'] / max(RET_VS_PX['n'], 1) * 100:.3f}% of name-days)")
print("   shortfall is measured off the compounded return series, so those breaks cannot leak into it")

# B2 closed-form VWAP versus an explicit day-by-day fill loop
def brute_vwap(i_start, i_stop, target, col, f_):
    got = num = 0.0
    for d in range(i_start, i_stop + 1):
        take = min(f_ * VOLMAT[d, col], target - got)
        if take <= 0:
            break
        num += take * PREL[d, col]; got += take
        if got >= target - 1e-6:
            break
    return (num / got if got > 0 else np.nan), got

worst = 0.0; checked = 0
for m in [len(FORMS) // 4, len(FORMS) // 2, 3 * len(FORMS) // 4]:
    d1 = FORMS[m]; i_dec = int(DPOS[d1]); dw = WVEC[d1] - WVEC[FORMS[m - 1]]
    res = month_paths(dw, BUCKET[d1], i_dec, BASE_LAG, BASE_CAP)
    if res is None:
        continue
    act, start, finish, bk, need = res
    P = price_paths(res, dw, m, i_dec, BASE_CAP)
    f_ = BASE_CAP / (1 - BASE_CAP)
    i_end = np.where(finish >= 0, np.maximum(finish, start), np.minimum(start + MAX_EXEC_DAYS, len(CAL) - 1))
    for j in range(len(act)):
        if bool(P["truncated"].iloc[j]) or float(P["fill_ratio"].iloc[j]) <= 0:
            continue
        bv, _got = brute_vwap(int(start[j]), int(i_end[j]), float(need[j] * f_), int(act[j]), f_)
        cf = float(P["vwap_rel"].iloc[j])
        if np.isfinite(bv) and bv > 0:
            worst = max(worst, abs(bv - cf) / bv); checked += 1
print(f"\nB2 closed-form VWAP versus a brute-force day loop, {checked} name-months: "
      f"max relative difference {worst:.2e}  (must be < 1e-9)")
assert worst < 1e-9, "the closed-form execution price does not reproduce an explicit fill loop"

# B3 start on the decision day and finish the same day -> zero delay, exactly
_D0 = run_combo(0, 0.30, detail=True)[1]
_one = _D0[(_D0["days_exec"] == 1) & (_D0["start_offset_bd"] == 0) & (~_D0["truncated"])]
_wmax = float(np.nanmax(np.abs(_one["delay_bps"]))) if len(_one) else 0.0
print(f"\nB3 names starting and finishing on the decision day (lag 0, cap 30%): {len(_one)}, "
      f"max |delay| among them {_wmax:.2e}bp  (must be 0)")
assert _wmax < 1e-9

# B4 monotonicity and invariance
_lo, _hi = run_combo(BASE_LAG, 0.02), run_combo(BASE_LAG, 0.30)
print("\nB4 impact (bp) at cap 2% versus cap 30%:")
print(pd.DataFrame({"cap 2%": _lo["impact_bps"], "cap 30%": _hi["impact_bps"]}).round(3).to_string())
assert (_hi["impact_bps"].dropna() >= _lo["impact_bps"].dropna() - 1e-9).all(), "impact must rise with participation"
_a = run_combo(0, 0.05, detail=True)[1][["decision", "code", "half_spread_bps"]]
_b = run_combo(20, 0.25, detail=True)[1][["decision", "code", "half_spread_bps"]]
_m = _a.merge(_b, on=["decision", "code"], suffixes=("_a", "_b"))
_inv = bool(np.allclose(_m["half_spread_bps_a"], _m["half_spread_bps_b"], equal_nan=True))
print(f"   per-unit half-spread identical for the same name-month at (lag 0, cap 5%) and "
      f"(lag 20, cap 25%), over {len(_m):,} name-months: {_inv}")
assert _inv, "the half-spread depends only on the decision-date tick band and must not move with lag or cap"
_s0 = run_combo(0, BASE_CAP, detail=True)[1]["delay_bps"].std()
_s20 = run_combo(20, BASE_CAP, detail=True)[1]["delay_bps"].std()
print(f"   dispersion of the delay term, lag 0 -> lag 20: {_s0:.1f}bp -> {_s20:.1f}bp "
      f"({'rises: waiting adds risk, not just cost' if _s20 > _s0 else 'DOES NOT RISE - investigate'})")

# B5 the decomposition adds up
S_chk, D_chk = run_combo(BASE_LAG, BASE_CAP, detail=True)
_err = (D_chk[["delay_bps", "spread_bps", "impact_bps", "opportunity_bps"]].sum(axis=1)
        - D_chk["total_is_bps"]).abs().max()
print(f"\nB5 max |delay + spread + impact + opportunity - total| = {_err:.2e}bp")
assert _err < 1e-9

# B6 no lookahead
viol = 0
for order in ["liquid_first", "parallel", "illiquid_first"]:
    for L in LAG_GRID:
        for m in range(1, len(FORMS)):
            d1 = FORMS[m]; i_dec = int(DPOS[d1])
            r = month_paths(WVEC[d1] - WVEC[FORMS[m - 1]], BUCKET[d1], i_dec, L, 0.30, order)
            if r is None:
                continue
            viol += int((r[1] < i_dec + L).sum())
print(f"\nB6 name-months starting before decision + lag: {viol}  (must be 0). "
      f"ADV, sigma and the tick band are all read at the decision date off shifted windows")
assert viol == 0

# B7 one month, printed
m = len(FORMS) // 2; d1 = FORMS[m]; i_dec = int(DPOS[d1]); dw = WVEC[d1] - WVEC[FORMS[m - 1]]
res = month_paths(dw, BUCKET[d1], i_dec, BASE_LAG, BASE_CAP)
EX = price_paths(res, dw, m, i_dec, BASE_CAP)
EX["bucket"] = [BUCKET_NAMES[b] for b in EX["bucket"]]
EX["start_date"] = CAL[res[1]]
print(f"\nB7 worked example  decision {d1.date()}  ->  earliest allowed trade {CAL[i_dec + BASE_LAG].date()} "
      f"(lag {BASE_LAG}bd, cap {BASE_CAP:.0%}, liquid-first)")
_ex = EX.sort_values(["bucket", "trade_yen"], ascending=[False, False]).groupby("bucket", observed=True).head(2)[
    ["code", "bucket", "side", "trade_yen", "start_date", "start_offset_bd", "days_exec", "fill_ratio",
     "delay_bps", "spread_bps", "impact_bps", "opportunity_bps", "total_is_bps"]]
print(_ex.round({c: 2 for c in _ex.select_dtypes("number").columns}).to_string(index=False))

## 5. Base case: what the schedule costs

Base combination is lag 2bd, cap 10%, liquid-first, `ETA` 0.5. Three views: the cost table by liquidity bucket with a month-clustered standard error on the total, the decomposition as a stacked bar, and — the question Day 3 handed forward — the same total under all three orderings, so liquid-first can finally be priced instead of described.

In [ ]:
S_base, D_base = run_combo(BASE_LAG, BASE_CAP, "liquid_first", detail=True)
print(f"base case: AUM {AUM:,.0f}, lag {BASE_LAG}bd, cap {BASE_CAP:.0%}, liquid-first, ETA {ETA}")
print(S_base.drop(columns=["order", "lag_bd", "cap"]).round(2).to_string())

_ok = D_base[~D_base["truncated"]]
_tot = vw(_ok, "total_is_bps")
_mo = vw_by(_ok, "total_is_bps", "decision").dropna()
_se = float(_mo.std(ddof=1) / np.sqrt(len(_mo)))
print(f"\nportfolio level: {_tot:.1f}bp per rebalance (month-clustered se {_se:.1f}, t {_tot / _se:.1f}), "
      f"about {_tot * 12 / 100:.2f}% a year at this turnover")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
comp = S_base[["delay_bps", "spread_bps", "impact_bps", "opportunity_bps"]].reindex(BUCKET_NAMES)
up = np.zeros(len(comp)); dn = np.zeros(len(comp))   # stack gains and costs separately: delay changes sign
for i, c in enumerate(comp.columns):
    v = comp[c].fillna(0).to_numpy()
    vp, vn = np.where(v > 0, v, 0.0), np.where(v < 0, v, 0.0)
    ax[0].bar(np.arange(5), vp, bottom=up, color=OI[i], label=c.replace("_bps", ""))
    ax[0].bar(np.arange(5), vn, bottom=dn, color=OI[i])
    up = up + vp; dn = dn + vn
ax[0].errorbar(np.arange(5), S_base["total_is_bps"].reindex(BUCKET_NAMES),
               yerr=1.96 * S_base["total_is_se"].reindex(BUCKET_NAMES), fmt="o", color="k", ms=4,
               lw=1, capsize=3, label="total (95% ci)")
ax[0].set_xticks(np.arange(5)); ax[0].set_xticklabels(BUCKET_NAMES, fontsize=8)
ax[0].axhline(0, lw=0.8, color="0.3"); ax[0].set_ylabel("basis points of target notional")
ax[0].set_title(f"Shortfall decomposition (lag {BASE_LAG}bd, cap {BASE_CAP:.0%})")
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.25, axis="y")

ORD = pd.concat([run_combo(BASE_LAG, BASE_CAP, o) for o in ["liquid_first", "parallel", "illiquid_first"]])
piv = ORD.reset_index().pivot(index="bucket", columns="order", values="total_is_bps").reindex(BUCKET_NAMES)
pse = ORD.reset_index().pivot(index="bucket", columns="order", values="total_is_se").reindex(BUCKET_NAMES)
pw  = ORD.reset_index().pivot(index="bucket", columns="order", values="sum_trade_yen").reindex(BUCKET_NAMES)
x = np.arange(5)
for i, o in enumerate(piv.columns):
    ax[1].bar(x + (i - 1) * 0.27, piv[o], width=0.27, color=OI[i], label=o,
              yerr=1.96 * pse[o], error_kw=dict(lw=0.8, capsize=2))
ax[1].set_xticks(x); ax[1].set_xticklabels(BUCKET_NAMES, fontsize=8)
ax[1].axhline(0, lw=0.8, color="0.3"); ax[1].set_ylabel("total shortfall (bp)")
ax[1].set_title("Does liquid-first pay?"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.25, axis="y")
fig.tight_layout(); fig.savefig(OUT_DIR / "day4_base_case.png", dpi=150); plt.show()

print("\ntotal shortfall (bp) by ordering:")
print(piv.round(2).to_string())
agg = {}
for o in piv.columns:
    s = piv[o].dropna()
    agg[o] = float(np.average(s, weights=pw[o].reindex(s.index).fillna(0) + 1e-9))
print("trade-weighted across buckets: " + ", ".join(f"{o} {v:.1f}bp" for o, v in agg.items()))
_gap = agg["liquid_first"] - agg["parallel"]
_lo_se = float(np.nanmean(pse["liquid_first"]))
print(f"\nliquid-first minus trading everything at once: {_gap:+.1f}bp, against a typical bucket "
      f"standard error of {_lo_se:.1f}bp.")
print("Read the sign, not the decimal: " + (
    "liquid-first is the more expensive schedule here, so the queueing discipline has to be justified by "
    "something this pass does not measure - risk, or the option to abandon the illiquid tail."
    if _gap > 0 else
    "liquid-first is the cheaper schedule here: the delay it adds to the illiquid tail costs less than the "
    "impact it saves by not pushing every name at once.") +
    " If the gap is inside the standard error, the two schedules are not distinguishable on cost alone.")

## 6. Full grid: lag x cap x liquidity, in basis points

The same `LAG_GRID` x `CAP_GRID` as Day 3, now scored on cost. The two axes pull in opposite directions, which is the whole point of the exercise:

- raising the **cap** shortens the execution window, so the delay term shrinks — but participation rises, and the square-root impact term grows. There should be an interior minimum, and it should sit in a different place for each liquidity bucket.
- raising the **lag** adds pure waiting. It cannot help capacity — Day 3 showed `days_exec` is flat in lag — so under any drift it moves the delay term monotonically, and it always widens that term's dispersion.

`ETA` is a modelling choice, not a measurement, so the last panel repeats the base column across `ETA_GRID`. If the cheapest cap moves with `ETA`, then the optimum is a property of the assumption and has to be quoted as a range.

In [ ]:
_t0 = time.time()
rows = []
for L in LAG_GRID:
    for c in CAP_GRID:
        rows.append(run_combo(L, c, "liquid_first"))
    print(f"  lag {L:>2}bd done ({time.time() - _t0:.0f}s)")
GRID = pd.concat(rows).reset_index()
GRID.to_csv(OUT_DIR / "day4_lag_cap_cost_grid.csv", index=False)
print(f"grid: {len(GRID)} rows in {time.time() - _t0:.0f}s -> {OUT_DIR / 'day4_lag_cap_cost_grid.csv'}")

for b in BUCKET_NAMES:
    print(f"\ntotal shortfall (bp), {b}   rows = cap, cols = lag(bd)")
    print(GRID[GRID["bucket"] == b].pivot(index="cap", columns="lag_bd", values="total_is_bps").round(1).to_string())

fig, axes = plt.subplots(1, 5, figsize=(22, 3.9))
vmax = float(np.nanmax(np.abs(GRID["total_is_bps"])))
for q, b in enumerate(BUCKET_NAMES):
    P = GRID[GRID["bucket"] == b].pivot(index="cap", columns="lag_bd", values="total_is_bps")
    axes[q].imshow(P.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    axes[q].set_xticks(range(len(P.columns))); axes[q].set_xticklabels(P.columns, fontsize=8)
    axes[q].set_yticks(range(len(P.index))); axes[q].set_yticklabels([f"{v:.0%}" for v in P.index], fontsize=8)
    axes[q].set_xlabel("lag (bd)"); axes[q].set_title(b, fontsize=10)
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            axes[q].text(j, i, f"{P.to_numpy()[i, j]:.0f}", ha="center", va="center", fontsize=6.5, color="k")
axes[0].set_ylabel("participation cap")
fig.suptitle("Total implementation shortfall, bp of target notional (red = costly)", y=1.03)
fig.tight_layout(); fig.savefig(OUT_DIR / "day4_is_heatmaps.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(18, 4.3))
for q, b in enumerate(BUCKET_NAMES):
    g = GRID[(GRID["bucket"] == b) & (GRID["lag_bd"] == BASE_LAG)].sort_values("cap")
    ax[0].plot(g["cap"] * 100, g["total_is_bps"], marker="o", color=OI[q], label=b)
    g2 = GRID[(GRID["bucket"] == b) & (GRID["cap"] == BASE_CAP)].sort_values("lag_bd")
    ax[1].plot(g2["lag_bd"], g2["total_is_bps"], marker="o", color=OI[q], label=b)
gq = GRID[(GRID["bucket"] == BUCKET_NAMES[0]) & (GRID["lag_bd"] == BASE_LAG)].sort_values("cap")
for i, c in enumerate(["delay_bps", "spread_bps", "impact_bps", "opportunity_bps"]):
    ax[2].plot(gq["cap"] * 100, gq[c], marker="o", color=OI[i], label=c.replace("_bps", ""))
ax[0].set_xlabel("participation cap (%)"); ax[0].set_ylabel("total shortfall (bp)")
ax[0].set_title(f"Cost versus cap (lag {BASE_LAG}bd)")
ax[1].set_xlabel("lag (bd)"); ax[1].set_ylabel("total shortfall (bp)")
ax[1].set_title(f"Cost versus lag (cap {BASE_CAP:.0%})")
ax[2].set_xlabel("participation cap (%)"); ax[2].set_ylabel("bp")
ax[2].set_title(f"{BUCKET_NAMES[0]}: what actually moves with the cap")
for a in ax:
    a.axhline(0, lw=0.8, color="0.3"); a.legend(fontsize=7); a.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(OUT_DIR / "day4_lag_cap_lines.png", dpi=150); plt.show()

print("\ncheapest (lag, cap) per bucket, against the base combination:")
best = []
for b in BUCKET_NAMES:
    g = GRID[GRID["bucket"] == b].dropna(subset=["total_is_bps"])
    if g.empty:
        continue
    r = g.loc[g["total_is_bps"].idxmin()]
    base = g[(g["lag_bd"] == BASE_LAG) & (g["cap"] == BASE_CAP)]["total_is_bps"]
    base = float(base.iloc[0]) if len(base) else np.nan
    best.append({"bucket": b, "best_lag_bd": int(r["lag_bd"]), "best_cap": f"{r['cap']:.0%}",
                 "best_bp": r["total_is_bps"], "base_bp": base, "saving_bp": base - r["total_is_bps"],
                 "se_bp": r["total_is_se"]})
print(pd.DataFrame(best).set_index("bucket").round(2).to_string())
print("A saving smaller than its own standard error is a grid artefact, not a schedule.")

ETAS = pd.concat([run_combo(BASE_LAG, c, "liquid_first", eta=e).assign(eta=e)
                  for e in ETA_GRID for c in CAP_GRID]).reset_index()
fig, ax = plt.subplots(figsize=(7.5, 4))
for i, e in enumerate(ETA_GRID):
    g = ETAS[(ETAS["eta"] == e) & (ETAS["bucket"] == BUCKET_NAMES[0])].sort_values("cap")
    ax.plot(g["cap"] * 100, g["total_is_bps"], marker="o", color=OI[i], label=f"ETA {e}")
ax.set_xlabel("participation cap (%)"); ax.set_ylabel("total shortfall (bp)")
ax.set_title(f"{BUCKET_NAMES[0]}: how much the optimum depends on the impact coefficient")
ax.axhline(0, lw=0.8, color="0.3"); ax.legend(fontsize=8); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(OUT_DIR / "day4_eta_sensitivity.png", dpi=150); plt.show()
print("\ncap that minimises Q1 cost, by impact coefficient:")
for e in ETA_GRID:
    g = ETAS[(ETAS["eta"] == e) & (ETAS["bucket"] == BUCKET_NAMES[0])].dropna(subset=["total_is_bps"])
    if not g.empty:
        print(f"  ETA {e:<5} -> cap {g.loc[g['total_is_bps'].idxmin(), 'cap']:.0%}")

## 7. Reading it, and what this deliberately does not do

**What the grid answers.** For a 1000億 fund running this monthly low-PBR list, what the month-end-to-completed-position round trip costs in basis points, split so that the four sources can be argued with separately, and where in the lag x cap plane that cost is smallest for each liquidity quintile. It also settles the question Day 3 handed forward: liquid-first versus trading everything at once is now a number with an error bar rather than a preference.

**Checks to make before quoting any of it:**
- Read `total_is_bps` next to `total_is_se`. The delay term is realised drift and its month-to-month standard deviation is an order of magnitude larger than its mean, so two buckets whose totals sit inside two standard errors of each other are not distinguishable, however different the point estimates look.
- `half_spread_bps` must be identical for a name-month across every lag and every cap. It depends only on the tick band at the decision date, so any variation means the ladder is being applied to the wrong price.
- Watch `fill_ratio` in Q1 at low caps alongside `opportunity_bps`. Once the fill ratio drops materially below 1, most of the reported "cost" is the cost of a position the fund never actually got — a capacity statement, not an execution statement.
- The optimum in the cap direction only means something if it survives `ETA_GRID`. If the cheapest cap moves from 2% to 30% between `ETA` 0.25 and 1.0, the honest report is a range, not a number.

**Known simplifications, all deliberate for this pass:**
- Spread and impact are **models, not measurements**. There is no quote or trade-print data in this file, so the spread is half of an assumed one-tick quote off the yobine ladder and impact is a square-root law with an assumed coefficient. Both are calibrations to be replaced when tick data is attached, and neither carries a standard error here, because their uncertainty is not sampling uncertainty.
- The tick grid is applied by date alone: the general ladder before 2023-06-05 and the fine TOPIX500 ladder from then on, that being the date the fine grid reached all of TOPIX500. TOPIX100 names carried the fine grid from 2014, so for the largest names before 2023 the spread term here is **too conservative** — it is the one term whose direction of error is known.
- Impact is charged as a pure temporary cost per slice and never decays into the price the fund's own later slices pay. A permanent-impact component would make large multi-day Q1 names look worse than they do here.
- The opportunity term marks an unfilled position at the cutoff price, so it is a paper number for a trade that never happened. It is reported separately for exactly that reason and should not be added into a quoted transaction cost.
- No intraday split and no clustering indicator, exactly as in Day 3. Both attach at the point where a day's slice is priced, which is now built.
- Trades are still the undrifted month-over-month weight change, so the trade list is Day 3's. A drift-consistent fund engine changes the sizes slightly, and therefore the impact term slightly, but not the shape of the grid.
- Participation caps still apply to the same day's printed volume, the same mild foresight Day 3 carried. An ADV-based cap is the conservative variant and is a one-line change.
- Volume is close times shares from the returns file, so it is a proxy for 売買代金, not the exchange's own figure.